## tl;dr
Medium fastest does not guarantee fastest in other SOC stages.
## Context & Methods
Five-drone, same-flight comparisons; first-drop SOC start; raw and frozen-Bideal-normalized rankings.
### Key Assumptions
Ranks are observational; small differences need not be statistically distinct.
## Data
Source files, model identity and hashes are in manifest.json.


## Results


In [1]:
from pathlib import Path
import sys,json
import pandas as pd
import numpy as np
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'battery_normalization.py').exists())
sys.path.insert(0,str(ROOT/'output_py'))
from audit_cross_stage_drone_rankings import calculate,OUT,digest
kept,ranks,comparisons,summary=calculate()
for source,expected in json.loads((OUT/'manifest.json').read_text())['source_sha256'].items():
    assert digest(ROOT/source)==expected
saved=pd.read_csv(OUT/'summary.csv')
assert np.allclose(summary.same_fastest,saved.same_fastest)
print(summary.to_string(index=False))
print(comparisons[comparisons['mode'].eq('Bideal')][['experiment_id','stage','medium_order','stage_order']].to_string(index=False))


stage   mode  flights  same_fastest  same_order  leader_still_top2  leader_within5pct_of_fastest  leader_within10pct_of_fastest  top_two_within2pct  median_rank_correlation  median_fastest_over_medium_leader_pct
 high Bideal        2             1           0                  2                             1                              1                   0                     0.05                              30.234856
 high    raw        2             0           0                  1                             0                              0                   0                    -0.65                              74.285639
  low Bideal       11             2           0                  3                             6                              9                   9                     0.10                               4.493546
  low    raw       11            10           0                 11                            11                             11                   2     

## Takeaways
Use actual stage rates rather than copying Medium ranks. Low top ranks are often near ties. No new experiments are requested.
Execution: cells ran sequentially in Python, not in a Jupyter kernel (nbformat/nbclient/ipykernel unavailable). To validate with Jupyter, run `python -m jupyter nbconvert --execute --to notebook --inplace ranking_audit.ipynb`.
